Daily Challenge : Text summarization using NLP


In [1]:
#Installation des Packages Nécessaires


!pip install pandas openpyxl nltk numpy scikit-learn networkx matplotlib
# Téléchargez les ressources NLTK nécessaires
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [22]:
import pandas as pd
from nltk.tokenize import sent_tokenize
import nltk
import numpy as np
import urllib.request
import zipfile
import os
import re
from nltk.corpus import stopwords
from string import punctuation
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx




In [4]:
# 1. Chargement et Inspection des Données

file_path = '/content/tennis_articles.csv'

try:
    df = pd.read_csv(file_path, encoding='latin1')
    print("Dataset chargé avec succès.")
    print("\nPremières lignes du dataset:")
    print(df.head())
    print("\nInformations sur le dataset:")
    df.info()

    # Supprimer la colonne 'article_title'
    if 'article_title' in df.columns:
        df = df.drop(columns=['article_title'])
        print("\nColonne 'article_title' supprimée.")
        print("\nNouvelles premières lignes après suppression:")
        print(df.head())
    else:
        print("\nLa colonne 'article_title' n'existe pas dans le dataset.")

except FileNotFoundError:
    print(f"Erreur: Le fichier '{file_path}' n'a pas été trouvé. Assurez-vous qu'il est téléversé dans Colab.")
except Exception as e:
    print(f"Une erreur est survenue lors du chargement ou de la manipulation du dataset: {e}")

# Travail avec la première ligne de 'article_text' pour ce défi.
# Sur le premier article pour le processus complet.
if 'df' in locals() and not df.empty and 'article_text' in df.columns:
    article_text = df['article_text'].iloc[0]
    print(f"\nArticle texte sélectionné pour le résumé (première entrée):")
    print(article_text[:500] + "...") # Affiche les 500 premiers caractères
else:
    article_text = ""
    print("Impossible de trouver 'article_text' ou le DataFrame est vide.")

Dataset chargé avec succès.

Premières lignes du dataset:
   article_id                                      article_title  \
0           1  I do not have friends in tennis, says Maria Sh...   
1           2  Federer defeats Medvedev to advance to 14th Sw...   
2           3  Tennis: Roger Federer ignored deadline set by ...   
3           4  Nishikori to face off against Anderson in Vien...   
4           5  Roger Federer has made this huge change to ten...   

                                        article_text  \
0  Maria Sharapova has basically no friends as te...   
1  BASEL, Switzerland (AP)  Roger Federer advanc...   
2  Roger Federer has revealed that organisers of ...   
3  Kei Nishikori will try to end his long losing ...   
4  Federer, 37, first broke through on tour over ...   

                                              source  
0  https://www.tennisworldusa.org/tennis/news/Mar...  
1  http://www.tennis.com/pro-game/2018/10/copil-s...  
2  https://scroll.in/field/8999

In [9]:
# 2. Tokenisation des Phrases


# Télécharger le ressource punkt_tab si nécessaire
nltk.download('punkt_tab')

if article_text:
    # Diviser l'article en phrases
    sentences = sent_tokenize(article_text, language='english')
    print(f"\nNombre de phrases extraites: {len(sentences)}")
    print("\nExemple de premières phrases:")
    for i, sent in enumerate(sentences[:5]):
        print(f"  {i+1}. {sent[:100]}...") # Affiche les 100 premiers caractères de chaque phrase
else:
    sentences = []
    print("Aucun texte d'article à traiter pour la tokenisation des phrases.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...



Nombre de phrases extraites: 19

Exemple de premières phrases:
  1. Maria Sharapova has basically no friends as tennis players on the WTA Tour....
  2. The Russian player has no problems in openly speaking about it and in a recent interview she said: '...
  3. I think everyone knows this is my job here....
  4. When I'm on the courts or when I'm on the court playing, I'm a competitor and I want to beat every s...
  5. So I'm not the one to strike up a conversation about the weather and know that in the next few minut...


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [15]:
# 3. Téléchargement et Chargement des Embeddings de Mots GloVe

# URL pour GloVe 6B 100d
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
glove_zip_path = "glove.6B.zip"
glove_txt_path = "glove.6B.100d.txt"


# Télécharger le fichier GloVe zip si non déjà présent or if it's a corrupted file
if not os.path.exists(glove_zip_path) or not zipfile.is_zipfile(glove_zip_path):
    if os.path.exists(glove_zip_path):
        print(f"Removing corrupted GloVe zip file: {glove_zip_path}")
        os.remove(glove_zip_path)
    print(f"Téléchargement de GloVe depuis {glove_url}...")
    urllib.request.urlretrieve(glove_url, glove_zip_path)
    print("Téléchargement terminé.")
else:
    print("Le fichier GloVe zip existe déjà et semble valide.")

# Dézipper le fichier GloVe si le fichier .txt n'existe pas
if not os.path.exists(glove_txt_path):
    print(f"Dézippage de {glove_zip_path}...")
    try:
        with zipfile.ZipFile(glove_zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Dézippage terminé.")
    except zipfile.BadZipFile:
        print(f"Erreur: Le fichier {glove_zip_path} est corrompu. Veuillez le supprimer manuellement et réessayer.")
else:
    print("Le fichier GloVe .txt existe déjà.")


# Charger les embeddings GloVe
word_embeddings = {}
try:
    with open(glove_txt_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            word_embeddings[word] = vector
    print(f"Chargement de {len(word_embeddings)} embeddings GloVe terminé.")
    if 'the' in word_embeddings:
        print(f"Dimension des embeddings: {word_embeddings['the'].shape[0]}")
    else:
        print("Word 'the' not found in embeddings.")
except FileNotFoundError:
    print(f"Erreur: Le fichier {glove_txt_path} n'a pas été trouvé. Assurez-vous qu'il est dézippé.")
except Exception as e:
    print(f"Une erreur est survenue lors du chargement des embeddings GloVe: {e}")

Removing corrupted GloVe zip file: glove.6B.zip
Téléchargement de GloVe depuis http://nlp.stanford.edu/data/glove.6B.zip...
Téléchargement terminé.
Dézippage de glove.6B.zip...
Dézippage terminé.
Chargement de 400000 embeddings GloVe terminé.
Dimension des embeddings: 100


In [18]:
# 4. Nettoyage et Normalisation du Texte

# Mots vides en anglais
stop_words = set(stopwords.words('english'))

def clean_sentence(sentence):
    # Convertir en minuscules
    sentence = sentence.lower()
    # Supprimer la ponctuation et les caractères spéciaux
    sentence = re.sub(r'[^\w\s]', '', sentence) # Garde seulement les mots et espaces
    sentence = re.sub(r'\d+', '', sentence) # Supprime les nombres
    # Tokeniser et supprimer les mots vides
    words = sentence.split()
    words = [word for word in words if word not in stop_words and word.strip() != '']
    return " ".join(words)

# Appliquer le nettoyage à toutes les phrases
cleaned_sentences = [clean_sentence(s) for s in sentences if s.strip()] # Évite les phrases vides

print(f"\nNombre de phrases nettoyées: {len(cleaned_sentences)}")
print("\nExemple de premières phrases nettoyées:")
for i, sent in enumerate(cleaned_sentences[:5]):
    print(f"  {i+1}. {sent[:100]}...")



Nombre de phrases nettoyées: 19

Exemple de premières phrases nettoyées:
  1. maria sharapova basically friends tennis players wta tour...
  2. russian player problems openly speaking recent interview said dont really hide feelings much...
  3. think everyone knows job...
  4. im courts im court playing im competitor want beat every single person whether theyre locker room ac...
  5. im one strike conversation weather know next minutes go try win tennis match...


In [19]:
# 5. Vectorisation des Phrases

# Définir la dimension des embeddings (100 pour glove.6B.100d.txt)
EMBEDDING_DIM = 100
# Créer un vecteur de zéros pour les mots inconnus
unknown_word_vector = np.zeros(EMBEDDING_DIM)

sentence_vectors = []
for sentence in cleaned_sentences:
    if not sentence.strip(): # Gérer le cas des phrases vides après nettoyage
        sentence_vectors.append(unknown_word_vector) # Ou simplement ignorer/ajouter un vecteur nul
        continue

    word_vectors = []
    for word in sentence.split():
        if word in word_embeddings:
            word_vectors.append(word_embeddings[word])
        else:
            word_vectors.append(unknown_word_vector) # Utilise un vecteur de zéros pour les mots inconnus

    if word_vectors: # S'assurer qu'il y a des vecteurs de mots pour éviter une erreur de division par zéro
        sentence_vectors.append(np.mean(word_vectors, axis=0))
    else:
        sentence_vectors.append(unknown_word_vector) # Cas où une phrase n'a aucun mot connu

# Convertir la liste de vecteurs en un tableau NumPy pour une manipulation plus facile
sentence_vectors = np.array(sentence_vectors)

print(f"\nForme des vecteurs de phrases: {sentence_vectors.shape}")
print("\nExemple du premier vecteur de phrase (premiers 5 éléments):")
print(sentence_vectors[0][:5])


Forme des vecteurs de phrases: (19, 100)

Exemple du premier vecteur de phrase (premiers 5 éléments):
[ 0.051489    0.1105585   0.6950863   0.18919174 -0.09581975]


In [21]:
# 6. Construction de la Matrice de Similarité

# Nous allons construire une matrice où chaque cellule (i,j) représente la similarité cosinus entre la phrase i et la phrase j.

# Initialiser une matrice vide de taille (nombre de phrases x nombre de phrases)
similarity_matrix = np.zeros((len(sentences), len(sentences)))

# Calculer la similarité cosinus entre chaque paire de vecteurs de phrases
# Attention: Assurez-vous que sentence_vectors est un tableau NumPy et non vide
if sentence_vectors.shape[0] > 0:
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i == j:
                continue # La similarité d'une phrase avec elle-même est 1, mais on ne l'utilise pas pour le graphe
            # Assurez-vous que les vecteurs ne sont pas tous nuls pour le calcul de similarité
            if np.all(sentence_vectors[i] == 0) or np.all(sentence_vectors[j] == 0):
                similarity_matrix[i][j] = 0 # Pas de similarité si un vecteur est nul
            else:
                # Reshape pour que cosine_similarity puisse traiter un seul échantillon
                similarity_matrix[i][j] = cosine_similarity(
                    sentence_vectors[i].reshape(1, -1),
                    sentence_vectors[j].reshape(1, -1)
                )[0,0]
    print("\nMatrice de similarité construite avec succès.")
    print(f"Forme de la matrice de similarité: {similarity_matrix.shape}")
    print("\nExemple de la matrice de similarité (premiers 5x5 éléments):")
    print(similarity_matrix[:5, :5])
else:
    print("\nAucun vecteur de phrase à traiter pour la matrice de similarité.")



Matrice de similarité construite avec succès.
Forme de la matrice de similarité: (19, 19)

Exemple de la matrice de similarité (premiers 5x5 éléments):
[[0.         0.6426971  0.59156992 0.72626793 0.7727943 ]
 [0.6426971  0.         0.85573618 0.81341243 0.83014321]
 [0.59156992 0.85573618 0.         0.78913021 0.79725081]
 [0.72626793 0.81341243 0.78913021 0.         0.88782728]
 [0.7727943  0.83014321 0.79725081 0.88782728 0.        ]]


In [23]:
# 7. Construction du Graphe et Classement des Phrases

# Créer un graphe à partir de la matrice de similarité
# On crée un graphe à partir de la matrice d'adjacence
graph = nx.from_numpy_array(similarity_matrix)

# Appliquer l'algorithme PageRank pour classer l'importance des phrases
# Le dictionnaire 'scores' contiendra le score PageRank pour chaque index de phrase
sentence_scores = nx.pagerank(graph)

print("\nGraphe construit et PageRank appliqué.")
print(f"Nombre de phrases classées: {len(sentence_scores)}")
# Afficher les 5 phrases les mieux classées (par leur index et score)
sorted_scores = sorted(sentence_scores.items(), key=lambda item: item[1], reverse=True)
print("\nTop 5 des scores PageRank (index de phrase, score):")
for i, (index, score) in enumerate(sorted_scores[:5]):
    print(f"  {i+1}. Index: {index}, Score: {score:.4f}")


Graphe construit et PageRank appliqué.
Nombre de phrases classées: 19

Top 5 des scores PageRank (index de phrase, score):
  1. Index: 15, Score: 0.0590
  2. Index: 9, Score: 0.0581
  3. Index: 3, Score: 0.0575
  4. Index: 4, Score: 0.0575
  5. Index: 7, Score: 0.0574


In [24]:
# 8. Résumé

# Enfin, nous trions les phrases en fonction de leurs scores PageRank décroissants et extrayons les N premières phrases pour former le résumé final.
#

# Trier les phrases originales par leur score PageRank en ordre décroissant
# Nous utilisons la liste 'sentences' originale pour le texte non modifié.
ranked_sentences_indices = [idx for idx, score in sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)]

# Définir le nombre de phrases à inclure dans le résumé
N = 10 # Par exemple, extraire les 10 phrases les plus importantes

# Extraire les top N phrases
summarized_sentences = [sentences[idx] for idx in ranked_sentences_indices[:N]]

print(f"\n--- Résumé Généré (Top {N} Phrases) ---")
for i, sentence in enumerate(summarized_sentences):
    print(f"{i+1}. {sentence}")

print("\n--- Fin du Résumé ---")


--- Résumé Généré (Top 10 Phrases) ---
1. I think everyone just thinks because we're tennis players we should be the greatest of friends.
2. When she said she is not really close to a lot of players, is that something strategic that she is doing?
3. When I'm on the courts or when I'm on the court playing, I'm a competitor and I want to beat every single person whether they're in the locker room or across the net.
4. So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match.
5. Uhm, I'm not really friendly or close to many players.
6. The Russian player has no problems in openly speaking about it and in a recent interview she said: 'I don't really hide any feelings too much.
7. I have not a lot of friends away from the courts.'
8. I think just because you're in the same sport doesn't mean that you have to be friends with everyone just because you're categorized, you're a tennis player, so you're goi

Autres possibilités d'optimisation 🇰


Autres modèles d'embeddings : FastText, Word2Vec, ou des embeddings contextuels comme BERT/RoBERTa.

Algorithmes de clustering : Pour regrouper des phrases similaires avant le classement.

Approches abstraites : Utiliser des modèles de génération (comme ceux de HuggingFace) pour créer un résumé "abstrait" plutôt qu'extractif.